<!-- # Training an optimal D-MPNN model

* Describe the details within a D-MPNN model
* Perform hyper-parameter optimization to find the best set of model parameters -->

In [ ]:
# import necessary libraries
import pickle
import os
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold

import torch
from lightning import pytorch as pl

from chemprop import data, featurizers, models, nn, utils
from chemprop.nn.metrics import RMSE, MSE

In [ ]:
# reducing verbosity
import logging
logging.getLogger("lightning.pytorch").setLevel(logging.ERROR)
logging.getLogger("pytorch_lightning").setLevel(logging.ERROR)
logging.getLogger("torch").setLevel(logging.ERROR)
import warnings
warnings.filterwarnings("ignore")

In [ ]:
# define a fixed random seed that is used to initialize randomizers used in the notebook
DATA_SPLIT_SEED = 42

pl.seed_everything(
    DATA_SPLIT_SEED,
    workers=True,
    verbose=True
)  # sets python/numpy/torch + dataloader workers

# Make PyTorch/Lightning more deterministic (esp. on GPU)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
# Ensure project root is on path (one level up from 'development')
PROJECT_ROOT = Path.cwd().parent
print(f"Project root: {PROJECT_ROOT}")

figs_dir = PROJECT_ROOT / "figs" # directory to save figures
print(f"Figures directory: {figs_dir}")

logs_dir = PROJECT_ROOT / "logs" / "model_tuning"
os.makedirs(logs_dir, exist_ok=True)

# Prepare dataset

In [ ]:
df_train = pd.read_csv(PROJECT_ROOT / "data" / "processed" / "training_data.csv")

In [ ]:
# Data columns
smiles_column = 'SMILES_clean' # Column containing SMILES strings
target_column = ['logTg'] # Column containing target property (log of glass transition temperature)

# List of molecular descriptors
descriptor_list = [
    'Moleculer Weight',
    'Proxy for Free Volume',
    'Rotatable Bonds',
    'Aromatic Rings',
    'Topological Polar Surface Area',
    'H-Bond Donors',
    'H-Bond Acceptors'
]
print(f"Total descriptors: {len(descriptor_list)}")

In [ ]:
# converting input data to numpy array for downstream processing
all_smis = df_train.loc[:, smiles_column].values
print("Training data input smiles array shape", all_smis.shape)
print(all_smis[:5])

In [ ]:
# converting additional descriptors data to numpy array
all_V_fs = df_train.loc[:, descriptor_list].values
print(all_V_fs.shape)

In [ ]:
# converting target data to numpy array
all_ys = df_train.loc[:, target_column].values
print("Target value array shape", all_ys.shape)
print(all_ys[:5])

In [ ]:
# create RDKit molecule objects from SMILES notation strings
all_mols = [
    utils.make_mol(smi, keep_h=False, add_h=False) for smi in all_smis
]

In [ ]:
K_FOLDS = 5

# KFold splitting
kf = KFold(
    n_splits=K_FOLDS,
    shuffle=True,
    random_state=DATA_SPLIT_SEED
)

# create dictionaries to hold train and validation indices for each fold
train_idx_dict = {}
val_idx_dict = {}
for fold, (train_idx, val_idx) in enumerate(kf.split(all_mols)):
    print(f"Fold {fold+1}")
    train_idx_dict[fold] = train_idx
    val_idx_dict[fold] = val_idx

# Model training

In [ ]:
# fixed configurations for model training
default_configs = {
    "train_seed_list": [42, 21, 7, 1, 10, 100, 5, 50, 80, 15],   # list of random seeds for model training
    "batch_norm": True, # whether to use batch normalization
    "batch_size": 64, # batch size for training
    "max_epochs": 100, # maximum number of training epochs
    "num_workers": 0, # number of workers for data loading
    "deterministic": True # Make PyTorch/Lightning more deterministic (esp. on GPU)
}

In [ ]:
# featurizer to transforms molecules into molecular graphs where atoms become nodes and bonds become edges
featurizer = featurizers.SimpleMoleculeMolGraphFeaturizer()

In [ ]:
# define hyperparameter options for tuning

# loss function names
loss_criteria = [
    RMSE(),
    MSE()
] 

ffn_dropout_options = [0.05, 0.1] # dropout rates in FFN
ffn_layers_options = [3, 4] # number of layers in FFN
ffn_hidden_dim_options = [200, 400] # hidden dimensions in FFN

mp_dropout_options = [0.05, 0.1] # dropout rates in message-passing network
mp_depth_options = [2, 4] # depth of message-passing network
mp_hidden_dim_options = [200, 400] # hidden dimensions in message-passing network

In [ ]:
def train_one_experiment(
    fold,
    all_mols,
    all_ys,
    train_idx,
    val_idx,
    default_configs,
    featurizer,
    logs_dir,
    tune_configs,
    all_V_fs=None,
):
    """Train one model with early stopping + best checkpoint and return a results row."""
    
    # initialize results list
    results = []

    
    for train_seed in default_configs["train_seed_list"]:
        print(f"Training with seed: {train_seed}")
        # Re-seed per run for reproducibility across a sweep
        pl.seed_everything(
            train_seed, workers=True, verbose=False
        )

        if all_V_fs is not None:
            train_data = [
                data.MoleculeDatapoint(
                    all_mols[train_idx[i]],
                    all_ys[train_idx[i]],
                    x_d=all_V_fs[train_idx[i]]
                ) for i in range(len(train_idx))
            ]
            val_data = [
                data.MoleculeDatapoint(
                    all_mols[val_idx[i]],
                    all_ys[val_idx[i]],
                    x_d=all_V_fs[val_idx[i]]
                ) for i in range(len(val_idx))
            ]
        else:
            train_data = [
                data.MoleculeDatapoint(
                    all_mols[train_idx[i]],
                    all_ys[train_idx[i]]
                ) for i in range(len(train_idx))
            ]
            val_data = [
                data.MoleculeDatapoint(
                    all_mols[val_idx[i]],
                    all_ys[val_idx[i]],
                ) for i in range(len(val_idx))
            ]

        train_dset = data.MoleculeDataset(train_data, featurizer)
        val_dset = data.MoleculeDataset(val_data, featurizer)
        
        # Normalize targets and descriptors
        fold_scaler = train_dset.normalize_targets()
        
        # Normalize validation targets with train set scaler
        val_dset.normalize_targets(fold_scaler)

        if all_V_fs is not None:
            fold_desc_scaler = train_dset.normalize_inputs("X_d")
            val_dset.normalize_inputs("X_d", fold_desc_scaler)
        

        train_loader = data.build_dataloader(
            train_dset,
            batch_size=default_configs["batch_size"],
            num_workers=default_configs["num_workers"],
            shuffle=True
        )
        val_loader = data.build_dataloader(
            val_dset,
            batch_size=default_configs["batch_size"],
            num_workers=default_configs["num_workers"],
            shuffle=False
        )

        # Model components
        ## Message passing
        mp = nn.BondMessagePassing()
        ## aggregation
        agg = nn.MeanAggregation()
        
        output_transform = nn.UnscaleTransform.from_standard_scaler(fold_scaler)
        
        if all_V_fs is not None:
            # Feed-forward network dimensions (message passing output + descriptor input)
            ffn_input_dim = mp.output_dim + all_V_fs.shape[1]
            # Descriptor input transformation
            X_d_transform = nn.ScaleTransform.from_standard_scaler(fold_desc_scaler)
        else:
            # Feed-forward network dimensions (message passing output only)
            ffn_input_dim = mp.output_dim
            # No descriptor input transformation
            X_d_transform = None

        # Feed-forward network
        ffn = nn.RegressionFFN(
            input_dim=ffn_input_dim,
            output_transform=output_transform,
            criterion=tune_configs["loss_criterion"],
        )
        # define mpnn model
        mpnn = models.MPNN(
            mp,
            agg,
            ffn,
            batch_norm=default_configs["batch_norm"],
            X_d_transform=X_d_transform
        )
        # Logging
        fold_logs_dir = Path(logs_dir) / f"fold_{fold+1}" / f"seed_{train_seed}"
        # initialize CSV logger
        csv_logger = pl.loggers.CSVLogger(
            save_dir=str(fold_logs_dir) if fold_logs_dir else ".",
            name="single_model"
        )

        # initialize PyTorch Lightning trainer
        trainer = pl.Trainer(
            logger=csv_logger,
            enable_checkpointing=False,
            enable_progress_bar=False,
            accelerator="auto",
            devices=1,
            max_epochs=default_configs["max_epochs"],
            deterministic=default_configs["deterministic"],
            enable_model_summary=False, # disables model summary printout
            log_every_n_steps=999999
        )

        # train the model
        trainer.fit(
            mpnn,
            train_loader,
            val_loader
        )

        # Save metrics path for later analysis
        metrics_path = Path(trainer.logger.log_dir) / "metrics.csv"
        
        # return results
        results.append(
            {
                "fold": fold+1,
                'train_seed': train_seed,
                "metrics_csv": str(metrics_path),
                "fold_scaling_factor": fold_scaler.scale_[0],
            }
        )

    return results

In [ ]:
def get_final_epoch_losses(results_dict, loss_criterion='RMSE'):
    train_losses = []
    val_losses = []
    for fold in results_dict:
        for run in results_dict[fold]:
            metrics_path = run['metrics_csv']
            df = pd.read_csv(metrics_path)
            df = df.sort_values(['epoch', 'step'])
            df_epoch = df.groupby('epoch', as_index=False).last()
            # If MSE, convert to RMSE
            if loss_criterion == 'MSE':
                train_loss = np.sqrt(df_epoch['train_loss_epoch'].values[-1])
                val_loss = np.sqrt(df_epoch['val_loss'].values[-1])
            else:
                train_loss = df_epoch['train_loss_epoch'].values[-1]
                val_loss = df_epoch['val_loss'].values[-1]
            train_losses.append(train_loss * run["fold_scaling_factor"])
            val_losses.append(val_loss * run["fold_scaling_factor"])
    return np.mean(train_losses), np.std(train_losses), np.mean(val_losses), np.std(val_losses)


def get_final_epoch_losses_early_stopping(
    results_dict,
    loss_criterion='RMSE',
    patience=20
):
    """
    Get train and val losses at early stopping epoch from results dictionary.
    
    Parameters:
    - results_dict (dict): Dictionary containing results for each fold and run.
    - loss_criterion (str): Loss criterion used ('RMSE' or 'MSE').
    - patience (int): Number of epochs to look back for early stopping.

    Returns:
    - Tuple: (mean_train_loss, std_train_loss, mean_val_loss, std_val_loss)
    """
    # initialize lists to hold train and val losses
    train_losses = []
    val_losses = []
    stop_epochs = []

    # Iterate through each fold and run to find early stopping epoch and corresponding losses
    for fold in results_dict:
        for run in results_dict[fold]:
            # Load metrics CSV
            metrics_path = run['metrics_csv']
            df = pd.read_csv(metrics_path)
            # Sort by epoch and step
            df = df.sort_values(['epoch', 'step'])
            # Group by epoch to get last entry per epoch
            df_epoch = df.groupby('epoch', as_index=False).last()
            
            # Find best epoch based on validation loss
            val_loss_arr = df_epoch['val_loss'].values
            best_idx = np.argmin(val_loss_arr)
            
            # Only allow early stop if best is at least 'patience' epochs before end
            if best_idx > len(val_loss_arr) - patience:
                best_idx = len(val_loss_arr) - patience
            # Record early stopping epoch
            stop_epochs.append(df_epoch['epoch'].values[best_idx])
            
            # Get corresponding train and val losses
            if loss_criterion == 'MSE': # convert to RMSE if MSE was used as loss criterion
                train_loss = np.sqrt(df_epoch['train_loss_epoch'].values[best_idx])
                val_loss = np.sqrt(df_epoch['val_loss'].values[best_idx])
            else:
                train_loss = df_epoch['train_loss_epoch'].values[best_idx]
                val_loss = df_epoch['val_loss'].values[best_idx]
            
            # Append losses rescaled to original units
            train_losses.append(train_loss * run["fold_scaling_factor"])
            val_losses.append(val_loss * run["fold_scaling_factor"])
    
    # Convert stop_epochs list to numpy array for statistics
    stop_epochs = np.array(stop_epochs)

    print(f"Early stopping epoch stats (patience={patience}):")
    print(
        f"Mean: {stop_epochs.mean():.1f}, Std: {stop_epochs.std():.1f}, Min: {stop_epochs.min()}, Max: {stop_epochs.max()}"
    )
    
    return np.mean(train_losses), np.std(train_losses), np.mean(val_losses), np.std(val_losses)

## Default Chemprop configurations

In [ ]:
tune_configs = {
    "loss_criterion": RMSE(),
    "ffn_dropout": 0,
    "ffn_layers": 2,
    "ffn_hidden_dim": 300,
    "mp_dropout": 0,
    "mp_depth": 3,
    "mp_hidden_dim": 300
}

In [ ]:
default_w_desc_log_dir = logs_dir / "default_w_desc_rmse"
os.makedirs(default_w_desc_log_dir, exist_ok=True)

exp_result_default_w_desc = {}

for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    exp_result_default_w_desc[fold_] = train_one_experiment(
        fold=fold_,
        all_mols=all_mols,
        all_ys=all_ys,
        train_idx=train_idx_dict[fold_],
        val_idx=val_idx_dict[fold_],
        default_configs=default_configs,
        featurizer=featurizer,
        logs_dir=default_w_desc_log_dir,
        tune_configs=tune_configs,
        all_V_fs=all_V_fs,
    )

In [ ]:
# Save results dictionary to a pickle file
with open(default_w_desc_log_dir / "exp_result_default_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_default_w_desc, f)

# To load it back later:
# with open(default_w_desc_log_dir / "exp_result_default_w_desc.pkl", "rb") as f:
#     exp_result_default_w_desc = pickle.load(f)

In [ ]:
default_no_desc_log_dir = logs_dir / "default_no_desc_rmse"
os.makedirs(default_no_desc_log_dir, exist_ok=True)

exp_result_default_no_desc = {}

for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    exp_result_default_no_desc[fold_] = train_one_experiment(
        fold=fold_,
        all_mols=all_mols,
        all_ys=all_ys,
        train_idx=train_idx_dict[fold_],
        val_idx=val_idx_dict[fold_],
        default_configs=default_configs,
        featurizer=featurizer,
        logs_dir=default_no_desc_log_dir,
        tune_configs=tune_configs,
    )

In [ ]:
# Save results dictionary to a pickle file
with open(default_no_desc_log_dir / "exp_result_default_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_default_no_desc, f)

# To load it back later:
# with open(default_no_desc_log_dir / "exp_result_default_no_desc.pkl", "rb") as f:
#     exp_result_default_no_desc = pickle.load(f)

In [ ]:
tune_configs_mse = tune_configs.copy()
tune_configs_mse["loss_criterion"] = MSE()
tune_configs_mse

In [ ]:
default_w_desc_mse_log_dir = logs_dir / "default_w_desc_mse"
os.makedirs(default_w_desc_mse_log_dir, exist_ok=True)

exp_result_default_mse_w_desc = {}

for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    exp_result_default_mse_w_desc[fold_] = train_one_experiment(
        fold=fold_,
        all_mols=all_mols,
        all_ys=all_ys,
        train_idx=train_idx_dict[fold_],
        val_idx=val_idx_dict[fold_],
        default_configs=default_configs,
        featurizer=featurizer,
        logs_dir=default_w_desc_mse_log_dir,
        tune_configs=tune_configs_mse,
        all_V_fs=all_V_fs,
    )

In [ ]:
# Save results dictionary to a pickle file
with open(default_w_desc_mse_log_dir / "exp_result_default_w_desc_mse.pkl", "wb") as f:
    pickle.dump(exp_result_default_mse_w_desc, f)

# To load it back later:
# with open(default_w_desc_mse_log_dir / "exp_result_default_w_desc_mse.pkl", "rb") as f:
#     exp_result_default_mse_w_desc = pickle.load(f)

In [ ]:
default_no_desc_mse_log_dir = logs_dir / "default_no_desc_mse"
os.makedirs(default_no_desc_mse_log_dir, exist_ok=True)

exp_result_default_mse_no_desc = {}

for fold_ in range(K_FOLDS):
    print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
    exp_result_default_mse_no_desc[fold_] = train_one_experiment(
        fold=fold_,
        all_mols=all_mols,
        all_ys=all_ys,
        train_idx=train_idx_dict[fold_],
        val_idx=val_idx_dict[fold_],
        default_configs=default_configs,
        featurizer=featurizer,
        logs_dir=default_no_desc_mse_log_dir,
        tune_configs=tune_configs_mse,
    )

In [ ]:
# Save results dictionary to a pickle file
with open(default_no_desc_mse_log_dir / "exp_result_default_no_desc_mse.pkl", "wb") as f:
    pickle.dump(exp_result_default_mse_no_desc, f)

# To load it back later:
# with open(default_no_desc_mse_log_dir / "exp_result_default_no_desc_mse.pkl", "rb") as f:
#     exp_result_default_mse_no_desc = pickle.load(f)

### Learning curves - Default configs

In [ ]:
# visualizations

def plot_training_validation_curves(
    results_dict,
    title_suffix="",
    loss_criterion='RMSE',
    save_path=None
):
    """Plot training and validation curves from logs directory."""
    # Collect all metrics.csv paths
    metrics_paths = []
    all_train = []
    all_val = []
    all_train_scaled = []
    all_val_scaled = []
    for fold in results_dict:
        for run in results_dict[fold]:
            metrics_paths.append(run['metrics_csv'])
            
            df = pd.read_csv(run['metrics_csv'])
            df = df.sort_values(['epoch', 'step'])
            df_epoch = df.groupby('epoch', as_index=False).last()
            
            if loss_criterion == 'MSE':
                # Convert MSE to RMSE for plotting
                df_epoch['train_loss_epoch'] = np.sqrt(df_epoch['train_loss_epoch'].values)
                df_epoch['val_loss'] = np.sqrt(df_epoch['val_loss'].values)

            all_train.append(df_epoch['train_loss_epoch'].values)
            all_val.append(df_epoch['val_loss'].values)

            all_train_scaled.append(df_epoch['train_loss_epoch'].values * run["fold_scaling_factor"])
            all_val_scaled.append(df_epoch['val_loss'].values * run["fold_scaling_factor"])
    
    plt.figure(figsize=(10, 5))
    for i, scaling_ in enumerate(['standard_scaled', 'rescaled']):
        plt.subplot(1, 2, i+1)
        if scaling_ == 'standard_scaled':
            all_train_used = all_train
            all_val_used = all_val
            ylabel = "RMSE (standard scaled units)"
        else:
            all_train_used = all_train_scaled
            all_val_used = all_val_scaled
            ylabel = "RMSE (original units)"
    
        # Calculate mean and std deviation across runs
        mean_train = np.mean(all_train_used, axis=0)
        std_train = np.std(all_train_used, axis=0)
        mean_val = np.mean(all_val_used, axis=0)
        std_val = np.std(all_val_used, axis=0)
        
        # Plotting
        epochs = np.arange(1, len(mean_train) + 1)
        
        
        plt.plot(epochs, mean_train, label='Training RMSE', color='blue')
        plt.fill_between(epochs, mean_train - std_train, mean_train + std_train, alpha=0.2, color='blue')
        
        plt.plot(epochs, mean_val, label='Validation RMSE', color='orange')
        plt.fill_between(epochs, mean_val - std_val, mean_val + std_val, alpha=0.2, color='orange')
        plt.legend(fontsize=12)        
        plt.xlabel('Epoch', fontsize=12)
        plt.ylabel(ylabel, fontsize=12)
        plt.xticks(fontsize=12)
        plt.yticks(fontsize=12)
        plt.grid(True, alpha=0.2)

    plt.suptitle(f'{title_suffix} \n Learning Curve (mean ± std over folds and seeds)', fontsize=12)
    plt.tight_layout()
    if save_path: plt.savefig(save_path, dpi=400, bbox_inches='tight')
    
    plt.show()

In [ ]:
plot_training_validation_curves(
    results_dict=exp_result_default_w_desc,
    title_suffix="Default Configs + RMSE loss w/ Descriptors",
    save_path= figs_dir / "learning_curve_default_w_desc_rmse.png"
)

In [ ]:
plot_training_validation_curves(
    results_dict=exp_result_default_no_desc,
    title_suffix="Default Configs + RMSE loss w/o Descriptors",
    save_path= figs_dir / "learning_curve_default_no_desc_rmse.png"
)

In [ ]:
plot_training_validation_curves(
    results_dict=exp_result_default_mse_w_desc,
    loss_criterion='MSE',
    title_suffix="Default Configs + MSE loss w/ Descriptors",
    save_path= figs_dir / "learning_curve_default_w_desc_mse.png"
)

In [ ]:
plot_training_validation_curves(
    results_dict=exp_result_default_mse_no_desc,
    loss_criterion='MSE',
    title_suffix="Default Configs + MSE loss w/o Descriptors",
    save_path= figs_dir / "learning_curve_default_no_desc_mse.png"
)

In [ ]:
# RMSE w/ descriptors
rmse_train_w_desc, rmse_train_w_desc_std, rmse_val_w_desc, rmse_val_w_desc_std = get_final_epoch_losses(exp_result_default_w_desc, 'RMSE')
# RMSE w/o descriptors
rmse_train_no_desc, rmse_train_no_desc_std, rmse_val_no_desc, rmse_val_no_desc_std = get_final_epoch_losses(exp_result_default_no_desc, 'RMSE')
# MSE w/ descriptors
mse_train_w_desc, mse_train_w_desc_std, mse_val_w_desc, mse_val_w_desc_std = get_final_epoch_losses(exp_result_default_mse_w_desc, 'MSE')
# MSE w/o descriptors
mse_train_no_desc, mse_train_no_desc_std, mse_val_no_desc, mse_val_no_desc_std = get_final_epoch_losses(exp_result_default_mse_no_desc, 'MSE')

print("After 100 epochs (mean ± std, original units):")
print(f"RMSE w/ descriptors:      Train={rmse_train_w_desc:.4f}±{rmse_train_w_desc_std:.4f}  Val={rmse_val_w_desc:.4f}±{rmse_val_w_desc_std:.4f}")
print(f"RMSE w/o descriptors:     Train={rmse_train_no_desc:.4f}±{rmse_train_no_desc_std:.4f}  Val={rmse_val_no_desc:.4f}±{rmse_val_no_desc_std:.4f}")
print(f"MSE w/ descriptors:       Train={mse_train_w_desc:.4f}±{mse_train_w_desc_std:.4f}  Val={mse_val_w_desc:.4f}±{mse_val_w_desc_std:.4f}")
print(f"MSE w/o descriptors:      Train={mse_train_no_desc:.4f}±{mse_train_no_desc_std:.4f}  Val={mse_val_no_desc:.4f}±{mse_val_no_desc_std:.4f}")

In [ ]:
# Example usage:
rmse_train_w_desc, rmse_train_w_desc_std, rmse_val_w_desc, rmse_val_w_desc_std = get_final_epoch_losses_early_stopping(exp_result_default_w_desc, 'RMSE', patience=20)
rmse_train_no_desc, rmse_train_no_desc_std, rmse_val_no_desc, rmse_val_no_desc_std = get_final_epoch_losses_early_stopping(exp_result_default_no_desc, 'RMSE', patience=20)
mse_train_w_desc, mse_train_w_desc_std, mse_val_w_desc, mse_val_w_desc_std = get_final_epoch_losses_early_stopping(exp_result_default_mse_w_desc, 'MSE', patience=20)
mse_train_no_desc, mse_train_no_desc_std, mse_val_no_desc, mse_val_no_desc_std = get_final_epoch_losses_early_stopping(exp_result_default_mse_no_desc, 'MSE', patience=20)

print("With early stopping (patience=20, mean ± std, original units):")
print(f"RMSE w/ descriptors:      Train={rmse_train_w_desc:.4f}±{rmse_train_w_desc_std:.4f}  Val={rmse_val_w_desc:.4f}±{rmse_val_w_desc_std:.4f}")
print(f"RMSE w/o descriptors:     Train={rmse_train_no_desc:.4f}±{rmse_train_no_desc_std:.4f}  Val={rmse_val_no_desc:.4f}±{rmse_val_no_desc_std:.4f}")
print(f"MSE w/ descriptors:       Train={mse_train_w_desc:.4f}±{mse_train_w_desc_std:.4f}  Val={mse_val_w_desc:.4f}±{mse_val_w_desc_std:.4f}")
print(f"MSE w/o descriptors:      Train={mse_train_no_desc:.4f}±{mse_train_no_desc_std:.4f}  Val={mse_val_no_desc:.4f}±{mse_val_no_desc_std:.4f}")

## FFN dropout

In [ ]:
ffn_dropout_w_desc_log_dir = logs_dir / "ffn_dropout_w_desc"
os.makedirs(ffn_dropout_w_desc_log_dir, exist_ok=True)

ffn_dropout_no_desc_log_dir = logs_dir / "ffn_dropout_no_desc"
os.makedirs(ffn_dropout_no_desc_log_dir, exist_ok=True)

exp_result_ffn_dropout_w_desc = {}
exp_result_ffn_dropout_no_desc = {}

for ffn_dropout in ffn_dropout_options:
    print(f"FFN Dropout: {ffn_dropout}")
    tune_configs_ffn_dropout = tune_configs.copy()
    tune_configs_ffn_dropout["ffn_dropout"] = ffn_dropout
    print(tune_configs_ffn_dropout)
    
    log_dir_w_desc = ffn_dropout_w_desc_log_dir / f"ffn_dropout_{ffn_dropout}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_ffn_dropout_w_desc[ffn_dropout] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_dropout_w_desc[ffn_dropout][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_ffn_dropout,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = ffn_dropout_no_desc_log_dir / f"ffn_dropout_{ffn_dropout}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_ffn_dropout_no_desc[ffn_dropout] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_dropout_no_desc[ffn_dropout][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_ffn_dropout,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(ffn_dropout_w_desc_log_dir / "exp_result_ffn_dropout_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_dropout_w_desc, f)

# To load it back later:
# with open(ffn_dropout_w_desc_log_dir / "exp_result_ffn_dropout_w_desc.pkl", "rb") as f:
#     exp_result_ffn_dropout_w_desc = pickle.load(f)

with open(ffn_dropout_no_desc_log_dir / "exp_result_ffn_dropout_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_dropout_no_desc, f)  

# To load it back later:
# with open(ffn_dropout_no_desc_log_dir / "exp_result_ffn_dropout_no_desc.pkl", "rb") as f:
#     exp_result_ffn_dropout_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for ffn_dropout in ffn_dropout_options:
    print(f"FFN Dropout: {ffn_dropout}")
    results = exp_result_ffn_dropout_no_desc[ffn_dropout]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for ffn_dropout in ffn_dropout_options:
    print(f"FFN Dropout (w/ desc): {ffn_dropout}")
    results = exp_result_ffn_dropout_w_desc[ffn_dropout]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

## FFN layer num

In [ ]:
ffn_layers_w_desc_log_dir = logs_dir / "ffn_layers_w_desc"
os.makedirs(ffn_layers_w_desc_log_dir, exist_ok=True)

ffn_layers_no_desc_log_dir = logs_dir / "ffn_layers_no_desc"
os.makedirs(ffn_layers_no_desc_log_dir, exist_ok=True)

exp_result_ffn_layers_w_desc = {}
exp_result_ffn_layers_no_desc = {}

for ffn_layer_num in ffn_layers_options:
    print(f"FFN Layers: {ffn_layer_num}")
    tune_configs_ffn_layers = tune_configs.copy()
    tune_configs_ffn_layers["ffn_layers"] = ffn_layer_num
    print(tune_configs_ffn_layers)
    
    log_dir_w_desc = ffn_layers_w_desc_log_dir / f"ffn_layers_{ffn_layer_num}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_ffn_layers_w_desc[ffn_layer_num] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_layers_w_desc[ffn_layer_num][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_ffn_layers,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = ffn_layers_no_desc_log_dir / f"ffn_layers_{ffn_layer_num}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_ffn_layers_no_desc[ffn_layer_num] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_layers_no_desc[ffn_layer_num][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_ffn_layers,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(ffn_layers_w_desc_log_dir / "exp_result_ffn_layers_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_layers_w_desc, f)

# To load it back later:
# with open(ffn_layers_w_desc_log_dir / "exp_result_ffn_layers_w_desc.pkl", "rb") as f:
#     exp_result_ffn_layers_w_desc = pickle.load(f)
with open(ffn_layers_no_desc_log_dir / "exp_result_ffn_layers_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_layers_no_desc, f)  

# To load it back later:
# with open(ffn_layers_no_desc_log_dir / "exp_result_ffn_layers_no_desc.pkl", "rb") as f:
#     exp_result_ffn_layers_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for ffn_layer_num in ffn_layers_options:
    print(f"FFN Layers: {ffn_layer_num}")
    results = exp_result_ffn_layers_no_desc[ffn_layer_num]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for ffn_layer_num in ffn_layers_options:
    print(f"FFN Layers (w/ desc): {ffn_layer_num}")
    results = exp_result_ffn_layers_w_desc[ffn_layer_num]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

## FFN hidden dim

In [ ]:
ffn_hidden_dim_w_desc_log_dir = logs_dir / "ffn_hidden_dim_w_desc"
os.makedirs(ffn_hidden_dim_w_desc_log_dir, exist_ok=True)

ffn_hidden_dim_no_desc_log_dir = logs_dir / "ffn_hidden_dim_no_desc"
os.makedirs(ffn_hidden_dim_no_desc_log_dir, exist_ok=True)

exp_result_ffn_hidden_dim_w_desc = {}
exp_result_ffn_hidden_dim_no_desc = {}

for ffn_hidden_dim in ffn_hidden_dim_options:
    print(f"FFN Hidden Dim: {ffn_hidden_dim}")
    tune_configs_ffn_hidden_dim = tune_configs.copy()
    tune_configs_ffn_hidden_dim["ffn_hidden_dim"] = ffn_hidden_dim
    print(tune_configs_ffn_hidden_dim)
    
    log_dir_w_desc = ffn_hidden_dim_w_desc_log_dir / f"ffn_hidden_dim_{ffn_hidden_dim}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_ffn_hidden_dim_w_desc[ffn_hidden_dim] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_hidden_dim_w_desc[ffn_hidden_dim][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_ffn_hidden_dim,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = ffn_hidden_dim_no_desc_log_dir / f"ffn_hidden_dim_{ffn_hidden_dim}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_ffn_hidden_dim_no_desc[ffn_hidden_dim] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_ffn_hidden_dim_no_desc[ffn_hidden_dim][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_ffn_hidden_dim,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(ffn_hidden_dim_w_desc_log_dir / "exp_result_ffn_hidden_dim_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_hidden_dim_w_desc, f)

# To load it back later:
# with open(ffn_hidden_dim_w_desc_log_dir / "exp_result_ffn_hidden_dim_w_desc.pkl", "rb") as f:
#     exp_result_ffn_hidden_dim_w_desc = pickle.load(f)

with open(ffn_hidden_dim_no_desc_log_dir / "exp_result_ffn_hidden_dim_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_ffn_hidden_dim_no_desc, f)  

# To load it back later:
# with open(ffn_hidden_dim_no_desc_log_dir / "exp_result_ffn_hidden_dim_no_desc.pkl", "rb") as f:
#     exp_result_ffn_hidden_dim_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for ffn_hidden_dim in ffn_hidden_dim_options:
    print(f"FFN Hidden Dim: {ffn_hidden_dim}")
    results = exp_result_ffn_hidden_dim_no_desc[ffn_hidden_dim]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for ffn_hidden_dim in ffn_hidden_dim_options:
    print(f"FFN Hidden Dim (w/ desc): {ffn_hidden_dim}")
    results = exp_result_ffn_hidden_dim_w_desc[ffn_hidden_dim]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

## MP dropout

In [ ]:
mp_dropout_w_desc_log_dir = logs_dir / "mp_dropout_w_desc"
os.makedirs(mp_dropout_w_desc_log_dir, exist_ok=True)

mp_dropout_no_desc_log_dir = logs_dir / "mp_dropout_no_desc"
os.makedirs(mp_dropout_no_desc_log_dir, exist_ok=True)

exp_result_mp_dropout_w_desc = {}
exp_result_mp_dropout_no_desc = {}

for mp_dropout in mp_dropout_options:
    print(f"MP dropout: {mp_dropout}")
    tune_configs_mp_dropout = tune_configs.copy()
    tune_configs_mp_dropout["mp_dropout"] = mp_dropout
    print(tune_configs_mp_dropout)
    
    log_dir_w_desc = mp_dropout_w_desc_log_dir / f"mp_dropout_{mp_dropout}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_mp_dropout_w_desc[mp_dropout] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_dropout_w_desc[mp_dropout][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_mp_dropout,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = mp_dropout_no_desc_log_dir / f"mp_dropout_{mp_dropout}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_mp_dropout_no_desc[mp_dropout] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_dropout_no_desc[mp_dropout][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_mp_dropout,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(mp_dropout_w_desc_log_dir / "exp_result_mp_dropout_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_dropout_w_desc, f)

# To load it back later:
# with open(mp_dropout_w_desc_log_dir / "exp_result_mp_dropout_w_desc.pkl", "rb") as f:
#     exp_result_mp_dropout_w_desc = pickle.load(f)

with open(mp_dropout_no_desc_log_dir / "exp_result_mp_dropout_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_dropout_no_desc, f)  

# To load it back later:
# with open(mp_dropout_no_desc_log_dir / "exp_result_mp_dropout_no_desc.pkl", "rb") as f:
#     exp_result_mp_dropout_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for mp_dropout in mp_dropout_options:
    print(f"MP Dropout: {mp_dropout}")
    results = exp_result_mp_dropout_no_desc[mp_dropout]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for mp_dropout in mp_dropout_options:
    print(f"MP Dropout (w/ desc): {mp_dropout}")
    results = exp_result_mp_dropout_w_desc[mp_dropout]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

## MP hidden dim

In [ ]:
mp_hidden_dim_w_desc_log_dir = logs_dir / "mp_hidden_dim_w_desc"
os.makedirs(mp_hidden_dim_w_desc_log_dir, exist_ok=True)

mp_hidden_dim_no_desc_log_dir = logs_dir / "mp_hidden_dim_no_desc"
os.makedirs(mp_hidden_dim_no_desc_log_dir, exist_ok=True)

exp_result_mp_hidden_dim_w_desc = {}
exp_result_mp_hidden_dim_no_desc = {}

for mp_hidden_dim in mp_hidden_dim_options:
    print(f"MP hidden dim: {mp_hidden_dim}")
    tune_configs_mp_hidden_dim = tune_configs.copy()
    tune_configs_mp_hidden_dim["mp_hidden_dim"] = mp_hidden_dim
    print(tune_configs_mp_hidden_dim)
    
    log_dir_w_desc = mp_hidden_dim_w_desc_log_dir / f"mp_hidden_dim_{mp_hidden_dim}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_mp_hidden_dim_w_desc[mp_hidden_dim] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_hidden_dim_w_desc[mp_hidden_dim][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_mp_hidden_dim,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = mp_hidden_dim_no_desc_log_dir / f"mp_hidden_dim_{mp_hidden_dim}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_mp_hidden_dim_no_desc[mp_hidden_dim] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_hidden_dim_no_desc[mp_hidden_dim][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_mp_hidden_dim,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(mp_hidden_dim_w_desc_log_dir / "exp_result_mp_hidden_dim_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_hidden_dim_w_desc, f)

# To load it back later:
# with open(mp_hidden_dim_w_desc_log_dir / "exp_result_mp_hidden_dim_w_desc.pkl", "rb") as f:
#     exp_result_mp_hidden_dim_w_desc = pickle.load(f)

with open(mp_hidden_dim_no_desc_log_dir / "exp_result_mp_hidden_dim_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_hidden_dim_no_desc, f)  

# To load it back later:
# with open(mp_hidden_dim_no_desc_log_dir / "exp_result_mp_hidden_dim_no_desc.pkl", "rb") as f:
#     exp_result_mp_hidden_dim_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for mp_hidden_dim in mp_hidden_dim_options:
    print(f"MP hidden dim: {mp_hidden_dim}")
    results = exp_result_mp_hidden_dim_no_desc[mp_hidden_dim]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for mp_hidden_dim in mp_hidden_dim_options:
    print(f"MP hidden dim (w/ desc): {mp_hidden_dim}")
    results = exp_result_mp_hidden_dim_w_desc[mp_hidden_dim]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

## MP depth

In [ ]:
mp_depth_w_desc_log_dir = logs_dir / "mp_depth_w_desc"
os.makedirs(mp_depth_w_desc_log_dir, exist_ok=True)

mp_depth_no_desc_log_dir = logs_dir / "mp_depth_no_desc"
os.makedirs(mp_depth_no_desc_log_dir, exist_ok=True)

exp_result_mp_depth_w_desc = {}
exp_result_mp_depth_no_desc = {}

for mp_depth in mp_depth_options:
    print(f"MP depth: {mp_depth}")
    tune_configs_mp_depth = tune_configs.copy()
    tune_configs_mp_depth["mp_depth"] = mp_depth
    print(tune_configs_mp_depth)
    
    log_dir_w_desc = mp_depth_w_desc_log_dir / f"mp_depth_{mp_depth}"
    os.makedirs(log_dir_w_desc, exist_ok=True)

    print("Model training With descriptors:")
    exp_result_mp_depth_w_desc[mp_depth] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_depth_w_desc[mp_depth][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_w_desc,
            tune_configs=tune_configs_mp_depth,
            all_V_fs=all_V_fs,
        )

    log_dir_no_desc = mp_depth_no_desc_log_dir / f"mp_depth_{mp_depth}"
    os.makedirs(log_dir_no_desc, exist_ok=True)
    
    
    print("Model training Without descriptors:")
    exp_result_mp_depth_no_desc[mp_depth] = {}
    
    for fold_ in range(K_FOLDS):
        print(f"Starting training for fold {fold_+1}/{K_FOLDS}")
        exp_result_mp_depth_no_desc[mp_depth][fold_] = train_one_experiment(
            fold=fold_,
            all_mols=all_mols,
            all_ys=all_ys,
            train_idx=train_idx_dict[fold_],
            val_idx=val_idx_dict[fold_],
            default_configs=default_configs,
            featurizer=featurizer,
            logs_dir=log_dir_no_desc,
            tune_configs=tune_configs_mp_depth,
        )

In [ ]:
# Save results dictionary to a pickle file
with open(mp_depth_w_desc_log_dir / "exp_result_mp_depth_w_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_depth_w_desc, f)

# To load it back later:
# with open(mp_depth_w_desc_log_dir / "exp_result_mp_depth_w_desc.pkl", "rb") as f:
#     exp_result_mp_depth_w_desc = pickle.load(f)

with open(mp_depth_no_desc_log_dir / "exp_result_mp_depth_no_desc.pkl", "wb") as f:
    pickle.dump(exp_result_mp_depth_no_desc, f)  

# To load it back later:
# with open(mp_depth_no_desc_log_dir / "exp_result_mp_depth_no_desc.pkl", "rb") as f:
#     exp_result_mp_depth_no_desc = pickle.load(f)

In [ ]:
# Summarize results for each FFN dropout value (no descriptors)
for mp_depth in mp_depth_options:
    print(f"MP Depth: {mp_depth}")
    results = exp_result_mp_depth_no_desc[mp_depth]
    # Last epoch
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    # Early stopping
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Repeat for w/ descriptors if needed:
for mp_depth in mp_depth_options:
    print(f"MP Depth (w/ desc): {mp_depth}")
    results = exp_result_mp_depth_w_desc[mp_depth]
    train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
    print(f"  Last epoch:      Train={train_mean:.4f}±{train_std:.4f}  Val={val_mean:.4f}±{val_std:.4f}")
    train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
    print(f"  Early stopping:  Train={train_mean_es:.4f}±{train_std_es:.4f}  Val={val_mean_es:.4f}±{val_std_es:.4f}")

# Aggregate model tuning results in a summary table

In [ ]:
# Define all your hyperparameter options
agg_results = []

# Helper to add results to the list
def add_results(hyperparam, value, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es):
    agg_results.append({
        "hyperparameter": hyperparam,
        "value": value,
        "with_descriptors": desc,
        "train_rmse_last": train_mean,
        "train_rmse_last_std": train_std,
        "val_rmse_last": val_mean,
        "val_rmse_last_std": val_std,
        "train_rmse_earlystop": train_mean_es,
        "train_rmse_earlystop_std": train_std_es,
        "val_rmse_earlystop": val_mean_es,
        "val_rmse_earlystop_std": val_std_es,
    })

# FFN Dropout
for desc, exp_result in zip([False, True], [exp_result_ffn_dropout_no_desc, exp_result_ffn_dropout_w_desc]):
    for ffn_dropout in ffn_dropout_options:
        results = exp_result[ffn_dropout]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("ffn_dropout", ffn_dropout, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# FFN Layers
for desc, exp_result in zip([False, True], [exp_result_ffn_layers_no_desc, exp_result_ffn_layers_w_desc]):
    for ffn_layers in ffn_layers_options:
        results = exp_result[ffn_layers]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("ffn_layers", ffn_layers, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# FFN Hidden Dim
for desc, exp_result in zip([False, True], [exp_result_ffn_hidden_dim_no_desc, exp_result_ffn_hidden_dim_w_desc]):
    for ffn_hidden_dim in ffn_hidden_dim_options:
        results = exp_result[ffn_hidden_dim]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("ffn_hidden_dim", ffn_hidden_dim, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# MP Dropout
for desc, exp_result in zip([False, True], [exp_result_mp_dropout_no_desc, exp_result_mp_dropout_w_desc]):
    for mp_dropout in mp_dropout_options:
        results = exp_result[mp_dropout]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("mp_dropout", mp_dropout, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# MP Hidden Dim
for desc, exp_result in zip([False, True], [exp_result_mp_hidden_dim_no_desc, exp_result_mp_hidden_dim_w_desc]):
    for mp_hidden_dim in mp_hidden_dim_options:
        results = exp_result[mp_hidden_dim]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("mp_hidden_dim", mp_hidden_dim, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# MP Depth
for desc, exp_result in zip([False, True], [exp_result_mp_depth_no_desc, exp_result_mp_depth_w_desc]):
    for mp_depth in mp_depth_options:
        results = exp_result[mp_depth]
        train_mean, train_std, val_mean, val_std = get_final_epoch_losses(results, 'RMSE')
        train_mean_es, train_std_es, val_mean_es, val_std_es = get_final_epoch_losses_early_stopping(results, 'RMSE', patience=20)
        add_results("mp_depth", mp_depth, desc, train_mean, train_std, val_mean, val_std, train_mean_es, train_std_es, val_mean_es, val_std_es)

# Save to DataFrame and CSV
agg_df = pd.DataFrame(agg_results)
agg_df.to_csv(PROJECT_ROOT / "supplementary_hyperparam_results.csv", index=False)
print("Aggregated results saved to supplementary_hyperparam_results.csv")

In [ ]:
agg_df.head()